# Clean Literature Dataset

Cleaning of raw literature records into a standardized CSV without destroying raw_dataset.

### Pipeline Summary
- **Standardization**: Normalizes sample IDs, mill types, and units while dropping key findings/density columns.
- **Process Parameters**: Parses `rpm` and `Time` (hours) as nullable integers (`Int64`).
- **Mechanical Properties**:
  - Hardness: $\text{HV} = \text{GPa} \times \frac{100}{0.98} \approx \text{GPa} \times 102.04$
  - Strength: $\sigma_{\text{HV}} = 3.4 \times \text{HV}$
- **Missing / Range Values**: Multi-condition ranges and non-comparable entries are stored as explicit `NA`.

In [ ]:
from pathlib import Path
import re

import pandas as pd


LOCAL_PATH = Path("../Data/Literature Data/TiAl_Ball_Milling_raw_dataset.csv")
GITHUB_URL = (
    "Data/TiAl_Ball_Milling_raw_dataset.csv"
)
source = LOCAL_PATH if LOCAL_PATH.exists() else GITHUB_URL
raw = pd.read_csv(source)
print(f"Loaded dataset from: {source}")

In [ ]:

DROP_COLUMNS = [
    'key_findings', 'strength_type', 'Density', 'Density_value', 'density_values',
    # Strength is derived from normalized hardness under the requested 3.4× rule.
    'strength_MPa',
]
to_drop = [column for column in DROP_COLUMNS if column in raw.columns]
working = raw.drop(columns=to_drop).rename(columns={
    'record_id': 'sample_id',
    'mill_type_class': 'mill_type_raw',
    'alloy_composition': 'composition_detail',
    'BPR': 'bpr_reported',
    'Grain_size': 'grain_size_reported',
    'Hardness': 'hardness_reported',
})


def simple_sample_name(composition: object) -> object:
    """Return the primary Ti-Al composition in a consistent, concise form."""
    if pd.isna(composition):
        return pd.NA
    text = str(composition)

    # Handles the reported Ti-(25-75)Al series.
    range_match = re.search(r'Ti-\((\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?)\)Al', text, flags=re.I)
    if range_match:
        return f"TiAl-{re.sub(r'\s+', '', range_match.group(1))}"

    # Uses the first named Ti-xAl composition as the concise sample name.
    # `composition_detail` retains all additions, alternatives, and unit notes.
    single_match = re.search(
        r'Ti-(\d+(?:\.\d+)?)\s*(?:at\.?%?\s*)?Al', text, flags=re.I
    )
    if single_match:
        al_content = single_match.group(1).rstrip('0').rstrip('.') if '.' in single_match.group(1) else single_match.group(1)
        return f'TiAl-{al_content}'
    return pd.NA


def clean_mill_type(value: object) -> tuple[object, object]:
    """Standardize the machine class and separate dry/wet descriptors from mill type."""
    if pd.isna(value):
        return pd.NA, pd.NA
    base = str(value).split('(')[0].strip()
    base = {'Planetry': 'Planetary'}.get(base, base)
    if base in {'Dry', 'Wet'}:
        return pd.NA, base
    return base, pd.NA


def parse_rpm(value: object) -> tuple[object, str]:
    """Parse a single reported RPM or leave a non-comparable/range value missing."""
    if pd.isna(value):
        return pd.NA, 'missing'
    text = str(value).strip()
    optimal = re.search(r'Optimal:\s*(\d+(?:\.\d+)?)\s*rpm', text, flags=re.I)
    if optimal:
        return int(round(float(optimal.group(1)))), 'reported_optimal'
    # Some records use a bare number in the RPM column; the column header supplies its unit.
    exact = re.fullmatch(r'(\d+(?:\.0+)?)(?:\s*rpm)?', text, flags=re.I)
    if exact:
        return int(float(exact.group(1))), 'exact'
    if 'intensity' in text.lower() or 'not comparable' in text.lower():
        return pd.NA, 'non_rpm_scale'
    return pd.NA, 'range_or_unparseable'


def parse_time_minutes(value: object) -> tuple[object, str]:
    """Convert a single duration to integer minutes without losing half-hour values."""
    if pd.isna(value):
        return pd.NA, 'missing'
    text = str(value).strip()
    optimal = re.search(r'Optimal:\s*(\d+(?:\.\d+)?)\s*h', text, flags=re.I)
    if optimal:
        return int(round(float(optimal.group(1)) * 60)), 'reported_optimal'
    exact = re.fullmatch(r'(\d+(?:\.\d+)?)\s*h', text, flags=re.I)
    if exact:
        return int(round(float(exact.group(1)) * 60)), 'exact'
    return pd.NA, 'range_or_unparseable'


HV_PER_GPA = 100 / 0.98
STRENGTH_FACTOR = 3.4


def hardness_to_hv(value: object) -> object:
    """Normalize a single reported hardness measurement to HV."""
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()

    # A hardness interval spans multiple material conditions, so it has no single value.
    if re.search(r'\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?\s*GPa', text, flags=re.I):
        return pd.NA

    hv = re.search(r'(~?\s*\d+(?:\.\d+)?)\s*HV', text, flags=re.I)
    if hv:
        return round(float(hv.group(1).replace('~', '').strip()), 2)

    gpa = re.search(r'(~?\s*\d+(?:\.\d+)?)\s*GPa', text, flags=re.I)
    if gpa:
        return round(float(gpa.group(1).replace('~', '').strip()) * HV_PER_GPA, 2)
    return pd.NA


In [ ]:
mill_pairs = working['mill_type_raw'].map(clean_mill_type)
rpm_pairs = working['RPM'].map(parse_rpm)
time_pairs = working['Time'].map(parse_time_minutes)
hardness_hv = pd.Series(working['hardness_reported'].map(hardness_to_hv), dtype='Float64')

cleaned = pd.DataFrame({
    'sample_id': working['sample_id'].astype('string'),
    'sample_name': working['composition_detail'].map(simple_sample_name).astype('string'),
    'mill_type': pd.Series([pair[0] for pair in mill_pairs], dtype='string'),
    'rpm': pd.Series([pair[0] for pair in rpm_pairs], dtype='Int64'),
    'Time': pd.Series([pair[0] for pair in time_pairs], dtype='Int64'),
    'bpr_reported': working['bpr_reported'].astype('string'),
    'hardness_hv': hardness_hv,
    'strength_hv': (hardness_hv * STRENGTH_FACTOR).round(2).astype('Float64'),
})

# Validation guards: the raw file is unchanged and every input row is retained.
assert len(cleaned) == len(raw)
assert cleaned['sample_id'].is_unique
assert str(cleaned['rpm'].dtype) == 'Int64'
assert str(cleaned['Time'].dtype) == 'Int64'
assert not set(DROP_COLUMNS).intersection(cleaned.columns)
assert str(cleaned['hardness_hv'].dtype) == 'Float64'
assert str(cleaned['strength_hv'].dtype) == 'Float64'
OUTPUT_DROPS = {
    'composition_detail', 'milling_medium', 'rpm_parse_status',
    'milling_time_min', 'time_parse_status',
    'grain_size_reported', 'hardness_reported', 'strength_mpa', 'strength',
    'model_input_complete',
}
assert not OUTPUT_DROPS.intersection(cleaned.columns)

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(OUTPUT, index=False, na_rep='NA')

print(f'Created: {OUTPUT.relative_to(ROOT)}')
print(f'Rows retained: {len(cleaned)}')
complete_process_inputs = cleaned['rpm'].notna() & cleaned['Time'].notna() & cleaned['bpr_reported'].notna()
print(f'Rows with RPM, time, and BPR reported: {int(complete_process_inputs.sum())}')
cleaned.head()


## Data Interpretation & Rules

- **Conversions**:
  - Hardness: $\text{HV} = \text{GPa} \times \frac{100}{0.98} \approx \text{GPa} \times 102.04$
  - Strength: $\sigma_{\text{HV}} = 3.4 \times \text{HV}$ (derived)
- **Status & Missing Values**:
  - `reported_optimal`: Retains single best condition when source reported a multi-condition range.
  - Multi-condition ranges (e.g., hardness intervals): Stored as `NA` to avoid arbitrary averaging.
  - Non-comparable RPM / units: Stored as `NA`.
- **Scope**: No sphericity target included.
